# C2 - SWELL-KW MODEL TABLE + PRELIMINARY MODELS (step 7)

SWELL-KW only (**not mixed with StudentLife**). Preliminary and exploratory. Not validation of C2.

## Decisions for this experiment (keep them in the report)
1. **Target** = the experimental **Condition** of that minute, an objective label from the study design (not self-report):
   `y_pressure = 1` for **T** (time pressure) or **I** (email interruptions); `y_pressure = 0` for **N** (neutral).
   Condition **R** rows are **excluded** (its meaning is not confirmed; check the ICMI 2014 paper). The letters N/T/I are read from the dataset description and the overview file.
   This is a **work-pressure label, NOT a C2 capacity label**.
2. **Inputs** (phone/PC/wearable-feasible only): computer-interaction features (File A) + physiology (File D: HR, RMSSD, SCL). Face and Kinect posture are **not used** (not available from a phone).
3. **HR and RMSSD use 999 for "no valid value"** in the file. 999 is converted to missing (NaN). It is never used as a number.
4. **No leakage:** each row uses only the **previous 5 minutes** of that person (including the current minute). Nothing from the future. `Blok`, `Condition` and the participant id are never features.
5. **Personal baseline** (core C2 idea): `dev_x` = the person's window value minus the mean of that person's EARLIER windows (needs 10 earlier minutes).
6. **Important limit:** all minutes of one block share the same label, and the dataset has only **25 people x 3 blocks = 75 labelled blocks**. So results are fragile, and grouped-by-person evaluation is essential.

## Cell 1 - Load files A and D

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json
import numpy as np
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

SWELL_DIR = None
for dirpath, dirnames, filenames in os.walk('/content/drive/MyDrive'):
    if 'Behavioral-features - per minute.xlsx' in filenames:
        SWELL_DIR = Path(dirpath); break
assert SWELL_DIR is not None
OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs'); OUT_DIR.mkdir(parents=True, exist_ok=True)

A = pd.read_csv(SWELL_DIR / 'A - Computer interaction features (Ulog - All Features per minute)-Sheet_1.csv')
D = pd.read_csv(SWELL_DIR / 'D - Physiology features (HR_HRV_SCL - final).csv')
A = A.loc[:, ~A.columns.str.startswith('Unnamed')]
D = D.loc[:, ~D.columns.str.startswith('Unnamed')].rename(columns={'C': 'Blok'})
print('A:', A.shape, '| D:', D.shape)

## Cell 2 - Sanity checks before building anything
- Time order and condition order for 3 participants (so we understand the study flow).
- How many HR / RMSSD values are the 999 "missing" code.
- How well A and D line up row by row.

In [ ]:
def parse_ts(s):
    """'YYYYMMDDTHHMMSSmmm'. Some values lost their trailing zeros (e.g. '20121001T162'), so pad the time part to 9 digits."""
    s = s.astype(str)
    date, tpart = s.str.slice(0, 8), s.str.slice(9)
    return pd.to_datetime(date + 'T' + tpart.str.ljust(9, '0'), format='%Y%m%dT%H%M%S%f')

for name, d in (('A', A), ('D', D)):
    lens = d['timestamp'].astype(str).str.len().value_counts().to_dict()
    odd = d.loc[d['timestamp'].astype(str).str.len() != 18, 'timestamp'].astype(str).head(5).tolist()
    print(f'{name}: timestamp string lengths {lens} | examples of non-standard ones: {odd}')
    d['time'] = parse_ts(d['timestamp'])

# parse check: consecutive minutes of the same person should be 1 minute apart
for name, d in (('A', A), ('D', D)):
    gaps = d.sort_values(['PP', 'time']).groupby('PP')['time'].diff().dt.total_seconds().div(60).round(1)
    print(f'{name}: minutes between consecutive rows (share):', gaps.value_counts(normalize=True).head(6).round(3).to_dict())

print('Condition order in time (Blok, Condition, first minute, number of minutes) for PP1, PP2, PP3:')
for pp in ['PP1', 'PP2', 'PP3']:
    g = A[A['PP'] == pp].sort_values('time')
    seg = (g[['Blok', 'Condition']] != g[['Blok', 'Condition']].shift()).any(axis=1).cumsum()
    out = g.groupby(seg).agg(Blok=('Blok', 'first'), Condition=('Condition', 'first'), start=('time', 'min'), minutes=('time', 'size'))
    print('\n', pp); print(out.to_string(index=False))

print('\nShare of 999 ("no valid value") in physiology:')
for c in ['HR', 'RMSSD', 'SCL']:
    print(f'  {c}: {(D[c] == 999).mean():.3f}')
hr_bad = D.assign(bad=(D['HR'] == 999)).groupby('PP')['bad'].mean().round(2)
print('\nparticipants with >=90% invalid HR:', hr_bad[hr_bad >= 0.9].index.tolist())
print('participants with 0% invalid HR   :', hr_bad[hr_bad == 0].index.tolist())

keys = ['PP', 'timestamp']
m = A.merge(D[keys + ['HR', 'RMSSD', 'SCL']], on=keys, how='outer', indicator=True)
print('\nrow alignment A vs D on (PP, timestamp):', m['_merge'].value_counts().to_dict())
print('duplicate (PP, timestamp) rows in A:', int(A.duplicated(keys).sum()), '| in D:', int(D.duplicated(keys).sum()))

## Cell 3 - Clean and merge A + D
- 999 -> NaN for HR and RMSSD.
- Keep only minutes present in both files (inner join). The mismatch is reported above.

In [ ]:
D2 = D.copy()
for c in ['HR', 'RMSSD']:
    D2.loc[D2[c] == 999, c] = np.nan
D2 = D2.drop_duplicates(['PP', 'timestamp'])
A2 = A.drop_duplicates(['PP', 'timestamp']).copy()

# Some File A cells hold Excel error text such as '#VALUE!' instead of a number. Convert to NaN and REPORT them.
sn_cols = [c for c in A2.columns if c.startswith('Sn')]
bad_report = {}
for c in sn_cols:
    num = pd.to_numeric(A2[c], errors='coerce')
    n_bad = int((num.isna() & A2[c].notna()).sum())
    if n_bad:
        bad_report[c] = (n_bad, A2.loc[num.isna() & A2[c].notna(), c].astype(str).unique().tolist()[:3])
    A2[c] = num
print('non-numeric cells found and set to NaN (column: count, examples):', bad_report if bad_report else 'none')
print('dtypes OK:', all(str(A2[c].dtype).startswith(('float', 'int')) for c in sn_cols))

df = A2.merge(D2[['PP', 'timestamp', 'HR', 'RMSSD', 'SCL']], on=['PP', 'timestamp'], how='inner')
df = df.sort_values(['PP', 'time']).reset_index(drop=True)
print('merged minutes:', df.shape, '| participants:', df['PP'].nunique())
print('conditions:', df['Condition'].value_counts().to_dict())
print('missing fraction  HR: %.3f  RMSSD: %.3f  SCL: %.3f' % tuple(df[c].isna().mean() for c in ['HR', 'RMSSD', 'SCL']))

## Cell 4 - Window features from the PREVIOUS 5 minutes (past only) and the target
- A features (16 behaviour counts per minute): window **mean**; plus window **std** for key typing/mouse features.
- Physiology: window mean of HR, RMSSD, SCL (+ SCL std) and `hr_valid_frac` (share of valid HR minutes in the window).
- A window needs at least 3 minutes of data. Rows are labelled by the Condition of the current minute.

In [ ]:
A_FEATS = [c for c in A.columns if c.startswith('Sn')]
print('A features:', len(A_FEATS), A_FEATS)
WIN, MINP = 5, 3

def make_windows(g):
    g = g.sort_values('time')
    out = pd.DataFrame(index=g.index)
    r = g[A_FEATS].rolling(WIN, min_periods=MINP)
    mean = r.mean(); mean.columns = [f'{c}_mean' for c in A_FEATS]
    out = out.join(mean)
    for c in ['SnKeyStrokes', 'SnMouseAct', 'SnMouseDistance']:
        out[f'{c}_std'] = g[c].rolling(WIN, min_periods=MINP).std()
    out['HR_mean'] = g['HR'].rolling(WIN, min_periods=MINP).mean()
    out['RMSSD_mean'] = g['RMSSD'].rolling(WIN, min_periods=MINP).mean()
    out['SCL_mean'] = g['SCL'].rolling(WIN, min_periods=MINP).mean()
    out['SCL_std'] = g['SCL'].rolling(WIN, min_periods=MINP).std()
    out['hr_valid_frac'] = g['HR'].notna().astype(float).rolling(WIN, min_periods=MINP).mean()
    return out

feats = df.groupby('PP', group_keys=False).apply(make_windows)
tbl = pd.concat([df[['PP', 'Blok', 'Condition', 'time']], feats], axis=1)
before = len(tbl)
tbl = tbl.dropna(subset=['SnKeyStrokes_mean', 'SCL_mean'])          # window needs >= 3 minutes
print(f'rows with a full window: {len(tbl)} of {before}')

tbl = tbl[tbl['Condition'].isin(['N', 'T', 'I'])].copy()            # R excluded (meaning unconfirmed)
tbl['y_pressure'] = tbl['Condition'].isin(['T', 'I']).astype(int)
print('after excluding R:', len(tbl), '| participants:', tbl['PP'].nunique())
print('class balance (1 = T or I):\n', tbl['y_pressure'].value_counts().to_string())
print('per condition:', tbl['Condition'].value_counts().to_dict())
print('rows per participant (min/median/max):', tbl.groupby('PP').size().min(), tbl.groupby('PP').size().median(), tbl.groupby('PP').size().max())

## Cell 5 - Personal-baseline deviation features (`dev_`)
`dev_x` = window value minus the mean of that same person's EARLIER windows (needs 10 earlier minutes). Only looks backward in time.

In [ ]:
tbl = tbl.sort_values(['PP', 'time']).reset_index(drop=True)
DEV_BASE = ['SnKeyStrokes_mean', 'SnMouseAct_mean', 'SnMouseDistance_mean', 'SnAppChange_mean', 'SnErrorKeys_mean',
            'HR_mean', 'RMSSD_mean', 'SCL_mean']
for c in DEV_BASE:
    prev = tbl.groupby('PP')[c].transform(lambda s: s.shift(1).expanding(min_periods=10).mean())
    tbl[f'dev_{c}'] = tbl[c] - prev

FEATURES = [c for c in tbl.columns if c not in ('PP', 'Blok', 'Condition', 'time', 'y_pressure')]
assert not any(f in FEATURES for f in ('PP', 'Blok', 'Condition', 'y_pressure'))
print('number of features:', len(FEATURES))
print('fraction missing (features with any missing):')
miss = tbl[FEATURES].isna().mean()
print(miss[miss > 0].round(3).sort_values(ascending=False).to_string())
tbl.to_csv(OUT_DIR / 'swell_model_table.csv', index=False)
print('\nsaved swell_model_table.csv (derived data - keep it in Drive, not in Git)')

## Cell 6 - CHECKPOINT: does the table look right?
Quick look at group differences (mean of a few features per condition). If the stressor conditions do not differ at all from neutral, the models cannot learn much.

In [ ]:
show = ['SnKeyStrokes_mean', 'SnMouseAct_mean', 'SnAppChange_mean', 'SnErrorKeys_mean', 'HR_mean', 'RMSSD_mean', 'SCL_mean']
print(tbl.groupby('Condition')[show].mean().round(3).to_string())
print('\nrows per participant AND condition (first 8 participants):')
print(tbl.groupby(['PP', 'Condition']).size().unstack(fill_value=0).head(8).to_string())

## Cell 7 - Models with participant-grouped cross-validation (5 folds x 5 different layouts)
Models: majority baseline, Logistic Regression, Random Forest (150 trees). Preparation steps are inside the pipeline. **No participant is ever in train and test together.**
Feature sets: **A+D all**, **A only (computer)**, **D only (physiology)**, **raw (no dev_)**, **dev_ only**.
Rule decided in advance: "clearly above chance" if `mean AUC > shuffled-label mean + 2 x shuffled std`.
This cell takes several minutes.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

def make_model(name):
    if name == 'Majority baseline':
        return Pipeline([('clf', DummyClassifier(strategy='most_frequent'))])
    if name == 'Logistic Regression':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=2000, random_state=42))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1))])

D_FEATS = [f for f in FEATURES if re.match(r'(dev_)?(HR|RMSSD|SCL)', f) or f == 'hr_valid_frac']
A_ONLY = [f for f in FEATURES if f not in D_FEATS]
SETS = {
    'A+D all': FEATURES,
    'A only (computer)': A_ONLY,
    'D only (physiology)': D_FEATS,
    'raw (no dev_)': [f for f in FEATURES if not f.startswith('dev_')],
    'dev_ only': [f for f in FEATURES if f.startswith('dev_')],
}
for k, v in SETS.items(): print(f'{k}: {len(v)} features')

y, groups = tbl['y_pressure'], tbl['PP']
scoring = {'roc_auc': 'roc_auc', 'f1': make_scorer(f1_score, zero_division=0), 'accuracy': 'accuracy'}
SEEDS = [0, 1, 2, 3, 4]

def run(model_name, cols, labels_fn):
    res_auc, res_f1, res_acc = [], [], []
    for s in SEEDS:
        yy = labels_fn(s)
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
        r = cross_validate(make_model(model_name), tbl[cols], yy, groups=groups, cv=cv, scoring=scoring)
        res_auc += list(r['test_roc_auc']); res_f1 += list(r['test_f1']); res_acc += list(r['test_accuracy'])
    return np.array(res_auc), np.array(res_f1), np.array(res_acc)

real = lambda s: y
shuf = lambda s: pd.Series(np.random.RandomState(100 + s).permutation(y.values), index=y.index)

rows = []
for set_name, cols in SETS.items():
    for mname in ['Majority baseline', 'Logistic Regression', 'Random Forest']:
        a, f, c = run(mname, cols, real)
        row = {'feature set': set_name, 'model': mname, 'AUC mean': a.mean(), 'AUC std': a.std(),
               'F1 mean': f.mean(), 'acc mean': c.mean()}
        if mname != 'Majority baseline':
            sa, _, _ = run(mname, cols, shuf)
            row['shuffled AUC mean'] = sa.mean(); row['shuffled AUC std'] = sa.std()
        rows.append(row)
    print('done:', set_name)

res = pd.DataFrame(rows)
res['threshold'] = res['shuffled AUC mean'] + 2 * res['shuffled AUC std']
res['clearly above chance?'] = np.where(res['AUC mean'] > res['threshold'], 'YES', np.where(res['model'] == 'Majority baseline', '-', 'no'))
print(res.round(3).to_string(index=False))
res.round(4).to_csv(OUT_DIR / 'swell_model_results.csv', index=False)

## Cell 8 - Picture: real AUC vs shuffled control

In [ ]:
import matplotlib.pyplot as plt
r2 = res[res['model'] != 'Majority baseline'].reset_index(drop=True)
x = np.arange(len(r2))
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x, r2['AUC mean'], yerr=r2['AUC std'], capsize=3, color='tab:blue', alpha=0.8, label='real labels')
ax.scatter(x, r2['shuffled AUC mean'], color='grey', zorder=3, label='shuffled labels (control)')
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['feature set']}\n{r['model'].replace('Logistic Regression','LR').replace('Random Forest','RF')}" for _, r in r2.iterrows()], rotation=60, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC (grouped CV)'); ax.set_ylim(0.4, 1.0); ax.legend()
plt.tight_layout(); plt.savefig(OUT_DIR / 'swell_auc_comparison.png', dpi=120); plt.show()

## Cell 9 - Confusion matrix of the best real model (pooled over cross-validation)
Chooses the (feature set, model) with the highest mean AUC among the real models and shows what it gets right and wrong.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, classification_report
best = res[res['model'] != 'Majority baseline'].sort_values('AUC mean', ascending=False).iloc[0]
print('best:', best['feature set'], '|', best['model'], '| AUC', round(best['AUC mean'], 3))
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
pred = cross_val_predict(make_model(best['model']), tbl[SETS[best['feature set']]], y, groups=groups, cv=cv)
cm = confusion_matrix(y, pred, labels=[0, 1])
print('confusion matrix (rows = truth, cols = prediction; 0 = neutral, 1 = time pressure or interruptions):')
print(cm)
print(classification_report(y, pred, target_names=['neutral (0)', 'pressure T/I (1)'], zero_division=0, digits=3))

## Cell 10 - What to send back to Claude
Nothing: Claude reads the outputs directly from this Colab tab. Models are NOT saved for export in this step (only after the results are reviewed).